# Test qwen

In [1]:
%cd /mnt/data1tb/thangcn/datnv2

/mnt/data1tb/thangcn/datnv2


In [2]:
from vllm import LLM, SamplingParams
import json
from openai import OpenAI
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_service_info, rag_product_info, rag_doctor_info, qa_medical, qa_symptom, book_appointment

/home/duyhoang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-07 10:22:27 [__init__.py:239] Automatically detected platform cuda.


2025-05-07 10:22:28,327	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/mnt/data1tb/thangcn/datnv2/service/func_for_fc.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [3]:
import dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
from service.search_doc import hybrid_search

In [4]:
EMBED_MODEL = "nampham1106/bkcare-embedding" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'}
)


In [5]:
with open('/mnt/data1tb/thangcn/datnv2/prompts/tools.json', 'r') as f:
    function_schema = json.load(f)
    
available_functions = {tool['name']: globals()[tool['name']] 
                                for tool in function_schema}

# Cấu hình LLM với tool_choice
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    temperature=0.8,
    model="thang1943/Llama-3.1-8B-in-ViMed",
)

In [6]:
tools = [
    {
        "type": "function",
        "function": tool
    } for tool in function_schema
]

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'rag_service_info',
   'description': "Retrieves detailed information about one or more services based on the user's query.",
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The service or category for which detailed information is being queried. This can refer to one or more services.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'rag_product_info',
   'description': "Retrieves detailed information about one or more products based on the user's query.",
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The product or item for which detailed information is being queried. This can refer to one or more products.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'rag_doctor_info',
   'description': "Retrieves detailed information about one or more doctors based on the user'

In [8]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

chat_response = client.chat.completions.create(
    model="thang1943/Llama-3.1-8B-in-ViMed",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Để tránh bị lây nhiễm virus Ebola, cần chú ý những điều gì?"},
    ],
    tools=tools,
    tool_choice="auto"
)
print("Chat response:", chat_response)

Chat response: ChatCompletion(id='chatcmpl-414b9fd82a2f4b129a6c054d1f7c11db', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='chatcmpl-tool-abe4a2854c8b438ea6aa9deee0f877be', function=Function(arguments='{"query": "c\\u01b0\\u1ed3m l\\u00e0y nhi\\u1ec5m virus Ebola"}', name='qa_medical'), type='function')], reasoning_content=None), stop_reason=None)], created=1746588179, model='thang1943/Llama-3.1-8B-in-ViMed', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=36, prompt_tokens=916, total_tokens=952, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None)


In [12]:
tool_calls = chat_response.choices[0].message.tool_calls

In [14]:
for tool_call in tool_calls:
    function_name = tool_call.function.name
    print(function_name)
    function_args = json.loads(tool_call.function.arguments)
    print(function_args)

qa_medical
{'query': 'cưồm lày nhiễm virus Ebola'}


In [8]:
messages=[{"role": "user", "content": "Để tránh bị lây nhiễm virus Ebola, cần chú ý những điều gì?"}],

In [45]:
print(retriever)

retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b62e6d79900>, search_kwargs={'k': 10}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7b62e6d799f0>, k=10)] weights=[0.5, 0.5]


# LLM

In [13]:

user_prompt = "Hello"

messages = [
    {'role': 'user', 'content': user_prompt}
]

response = llm.predict_messages(
    messages,
    tools=tools,
)

In [14]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-f56010f0111447bcb72f25716eb75ee8', 'function': {'arguments': '{"query": "Hello"}', 'name': 'rag_service_info'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 957, 'total_tokens': 975, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'thang1943/Llama-3.1-8B-in-ViMed', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-ac9bbb6e-5996-4ea3-af72-6c2883b618e8-0', tool_calls=[{'name': 'rag_service_info', 'args': {'query': 'Hello'}, 'id': 'chatcmpl-tool-f56010f0111447bcb72f25716eb75ee8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 957, 'output_tokens': 18, 'total_tokens': 975, 'input_token_details': {}, 'output_token_details': {}})

In [15]:
response.additional_kwargs['tool_calls']

[{'id': 'chatcmpl-tool-f56010f0111447bcb72f25716eb75ee8',
  'function': {'arguments': '{"query": "Hello"}', 'name': 'rag_service_info'},
  'type': 'function'}]

In [16]:
tool_calls = response.additional_kwargs['tool_calls']

In [17]:
for tool in tool_calls:
    function_name = tool['function']['name']
    function_args = tool['function']['arguments']
    function_args = function_args.encode('utf-8').decode('unicode_escape')
    print(f"Function name: {function_name}")
    print(f"Function arguments: {function_args}")

Function name: rag_service_info
Function arguments: {"query": "Hello"}


In [18]:
function_args = json.loads(function_args)

In [19]:
retriever = available_functions[function_name](**function_args)

In [20]:
retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f4d3a119660>, search_kwargs={'k': 10}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f4d3a119750>, k=10)], weights=[0.5, 0.5])

In [21]:
retriever.get_relevant_documents("Nước súc miệng Listerine Mint 70ml")

/tmp/ipykernel_1866518/3355701637.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retriever.get_relevant_documents("Nước súc miệng Listerine Mint 70ml")


[Document(metadata={}, page_content='Mã tương đương: 14.0210.0799, Tên dịch vụ kỹ thuật theo Thông tư 23/2024/TT-BYT: Nặn tuyến bờ mi, đánh bờ mi, Tên dịch vụ phê duyệt giá: Nặn tuyến bờ mi, đánh bờ mi, Phân Loại PTTT: T3, Mức giá: 40.900, Ghi chú: \n'),
 Document(id='3ebe6607-c35f-49fb-ae3f-b93b7781d0ff', metadata={}, page_content='Mã tương đương: 23.0246.1558, Tên dịch vụ kỹ thuật theo Thông tư 23/2024/TT-BYT: Định lượng Salicylate, Tên dịch vụ phê duyệt giá: Định lượng Salicylate, Phân Loại PTTT: , Mức giá: 78.500, Ghi chú: \n'),
 Document(metadata={}, page_content='Mã tương đương: 18.0098.0010, Tên dịch vụ kỹ thuật theo Thông tư 23/2024/TT-BYT: Chụp X-quang khung chậu thẳng, Tên dịch vụ phê duyệt giá: Chụp X-quang khung chậu thẳng [≤ 24x30 cm, 1 tư thế], Phân Loại PTTT: , Mức giá: 58.300, Ghi chú: Áp dụng cho 01 vị trí.\n'),
 Document(id='d250fa8e-4591-46c3-84f9-4a1359c44826', metadata={}, page_content='Mã tương đương: 03.4208.0302, Tên dịch vụ kỹ thuật theo Thông tư 23/2024/TT-BYT